# Fractal Baseline Training Guide

Purpose:
- train calibrated baseline classifiers for fractal swing labels
- export the exact artifacts needed by the online detector

Inputs:
- `ETHUSDT_15m_features_reduced.parquet`
- `ETHUSDT_15m_fractal_labels_L10_R10.parquet`
- optional clean OHLC parquet for reference

Outputs:
- saved high and low models
- `feature_config.json`
- `thresholds.json`
- `meta.json`

Reading guide:
1. Load and align features with fractal labels.
2. Define the explicit feature and target contract.
3. Split the timeline into train, validation, and test blocks.
4. Train calibrated baselines and choose thresholds on validation.
5. Run the final out-of-sample test and save all runtime artifacts.

Important note:
- Threshold selection is done on the validation split using confirmation-aware logic; the test split stays untouched until the final evaluation.


### Imports & constants

In [1]:
from __future__ import annotations

import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd

import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Config (paths, split, fractal)

In [ ]:
@dataclass(frozen=True)
class DataPaths:
    """Container for dataset file paths."""
    features_path: Path
    labels_path: Path
    ohlc_path: Optional[Path] = None


@dataclass(frozen=True)
class ArtifactPaths:
    """Container for saved model and config artifact paths."""
    model_high_path: Path
    model_low_path: Path
    feature_config_path: Path
    thresholds_path: Path
    meta_path: Path


@dataclass(frozen=True)
class FractalConfig:
    """Fractal labeling configuration."""
    left: int
    right: int


@dataclass(frozen=True)
class SplitConfig:
    """Chronological split configuration."""
    train_ratio: float = 0.70
    valid_ratio: float = 0.15
    test_ratio: float = 0.15


def locate_ml_root(start=None) -> Path:
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == "ML v1" and (candidate / "data").exists():
            return candidate

        ml_root = candidate / "ML v1"
        if (ml_root / "data").exists() and (ml_root / "code").exists():
            return ml_root

    raise FileNotFoundError(
        "Could not locate the 'ML v1' workspace from the current working directory."
    )


ML_ROOT = locate_ml_root()
PROJECT_ROOT = ML_ROOT.parent
DATA_DIR = ML_ROOT / "data"
CONFIG_DIR = ML_ROOT / "configs"
CODE_DIR = ML_ROOT / "code"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DATA_PATHS = DataPaths(
    features_path=DATA_DIR / "ETHUSDT_15m_features_reduced.parquet",
    labels_path=DATA_DIR / "ETHUSDT_15m_fractal_labels_L10_R10.parquet",
    ohlc_path=DATA_DIR / "ETHUSDT_15m_ohlc_clean.parquet",
)

ARTIFACT_PATHS = ArtifactPaths(
    model_high_path=DATA_DIR / "model_high.pkl",
    model_low_path=DATA_DIR / "model_low.pkl",
    feature_config_path=CONFIG_DIR / "feature_config.json",
    thresholds_path=CONFIG_DIR / "thresholds.json",
    meta_path=CONFIG_DIR / "meta.json",
)

FRACTAL_CFG = FractalConfig(left=10, right=10)
SPLIT_CFG = SplitConfig()

for output_path in (
    ARTIFACT_PATHS.model_high_path,
    ARTIFACT_PATHS.model_low_path,
    ARTIFACT_PATHS.feature_config_path,
    ARTIFACT_PATHS.thresholds_path,
    ARTIFACT_PATHS.meta_path,
):
    output_path.parent.mkdir(parents=True, exist_ok=True)


### IO & alignment utilities

In [3]:
def load_parquet(path: Path) -> pd.DataFrame:
    """
    Load a parquet file into a DataFrame.

    Args:
        path: parquet file path.

    Returns:
        Loaded dataframe.

    Raises:
        FileNotFoundError: if path does not exist.
    """
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return pd.read_parquet(path)


def infer_time_column(dataframe: pd.DataFrame) -> Optional[str]:
    """
    Infer a time column name from common candidates.

    Args:
        dataframe: input dataframe.

    Returns:
        Column name if found, otherwise None.
    """
    candidates = ["timestamp", "open_time", "time", "datetime", "date"]
    for name in candidates:
        if name in dataframe.columns:
            return name
    return None


def set_time_index(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Set a datetime index using an inferred time column if present.

    Args:
        dataframe: input dataframe.

    Returns:
        DataFrame with a datetime index (if possible).
    """
    time_col = infer_time_column(dataframe)
    if time_col is None:
        return dataframe

    df = dataframe.copy()
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce")
    df = df.dropna(subset=[time_col]).sort_values(time_col)
    df = df.set_index(time_col)
    return df


def align_features_and_labels(features: pd.DataFrame, labels: pd.DataFrame) -> pd.DataFrame:
    """
    Align features and labels by time index.

    Args:
        features: feature dataframe indexed by time.
        labels: labels dataframe indexed by time.

    Returns:
        Joined dataframe with features + label columns.
    """
    features_idx = set_time_index(features)
    labels_idx = set_time_index(labels)

    joined = features_idx.join(labels_idx, how="inner").sort_index()
    return joined

### Load data + basic contract checks

In [4]:
features_df = load_parquet(DATA_PATHS.features_path)
labels_df = load_parquet(DATA_PATHS.labels_path)

dataset = align_features_and_labels(features_df, labels_df)

print("Dataset shape:", dataset.shape)
print("Index range:", dataset.index.min(), "->", dataset.index.max())
print("Columns:", dataset.columns.tolist())

if not dataset.index.is_monotonic_increasing:
    raise ValueError("Index is not sorted increasing. Alignment issue.")

if dataset.index.has_duplicates:
    raise ValueError("Index has duplicates. Alignment issue.")

Dataset shape: (139219, 11)
Index range: 2021-07-05 12:00:00+00:00 -> 2025-06-28 20:30:00+00:00
Columns: ['ret_1', 'ret_3', 'body', 'upper_wick', 'lower_wick', 'dist_to_roll_max_20', 'dist_to_roll_min_20', 'vol_50', 'segment_id', 'y_high', 'y_low']


### Targets & features (explicit contract)

In [5]:
def infer_label_columns(df: pd.DataFrame) -> Tuple[str, str]:
    """
    Infer label column names for swing high and swing low.

    Args:
        df: merged dataset.

    Returns:
        Tuple of (high_label_col, low_label_col).

    Raises:
        ValueError: if columns cannot be inferred.
    """
    candidates = [
        ("y_high", "y_low"),
        ("swing_high", "swing_low"),
        ("fractal_high", "fractal_low"),
        ("label_high", "label_low"),
    ]
    for high_col, low_col in candidates:
        if high_col in df.columns and low_col in df.columns:
            return high_col, low_col

    raise ValueError("Could not infer label columns (high/low).")


high_label_col, low_label_col = infer_label_columns(dataset)

# Explicitly exclude non-features that should never be fed into the model
forbidden_cols = {high_label_col, low_label_col}

# If you have IDs/regime labels, decide explicitly whether to include them
# For baseline: exclude segment_id unless you deliberately want it.
if "segment_id" in dataset.columns:
    forbidden_cols.add("segment_id")

feature_cols = [c for c in dataset.columns if c not in forbidden_cols]

X_all = dataset[feature_cols].copy()
y_high_all = dataset[high_label_col].astype(int).copy()
y_low_all = dataset[low_label_col].astype(int).copy()

print("n_rows:", len(dataset))
print("n_features:", len(feature_cols))
print("feature_cols:", feature_cols)
print("positive_rate_high:", float(y_high_all.mean()))
print("positive_rate_low:", float(y_low_all.mean()))

n_rows: 139219
n_features: 8
feature_cols: ['ret_1', 'ret_3', 'body', 'upper_wick', 'lower_wick', 'dist_to_roll_max_20', 'dist_to_roll_min_20', 'vol_50']
positive_rate_high: 0.03373821102004755
positive_rate_low: 0.03424101595328224


### Chronological split

In [6]:
def time_split_indices(n_rows: int, split_cfg: SplitConfig) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Create chronological train/valid/test indices.

    Args:
        n_rows: number of rows.
        split_cfg: split configuration.

    Returns:
        train_idx, valid_idx, test_idx
    """
    train_end = int(n_rows * split_cfg.train_ratio)
    valid_end = int(n_rows * (split_cfg.train_ratio + split_cfg.valid_ratio))

    idx = np.arange(n_rows)
    return idx[:train_end], idx[train_end:valid_end], idx[valid_end:]


train_idx, valid_idx, test_idx = time_split_indices(len(dataset), SPLIT_CFG)

X_train = X_all.iloc[train_idx]
X_valid = X_all.iloc[valid_idx]
X_test = X_all.iloc[test_idx]

y_high_train = y_high_all.iloc[train_idx]
y_high_valid = y_high_all.iloc[valid_idx]
y_high_test = y_high_all.iloc[test_idx]

y_low_train = y_low_all.iloc[train_idx]
y_low_valid = y_low_all.iloc[valid_idx]
y_low_test = y_low_all.iloc[test_idx]

print("Splits:", len(X_train), len(X_valid), len(X_test))

Splits: 97453 20883 20883


### Models (baseline choices)

In [7]:
def build_hgb_model() -> HistGradientBoostingClassifier:
    """
    Build a histogram gradient boosting baseline.

    Returns:
        HistGradientBoostingClassifier instance.
    """
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        random_state=RANDOM_SEED,
    )


def build_logreg_model() -> Pipeline:
    """
    Build a logistic regression baseline pipeline.

    Returns:
        Sklearn Pipeline with scaler + logistic regression.
    """
    return Pipeline(
        steps=[
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )

### Sample weights

In [8]:
def compute_sample_weights(y: pd.Series) -> np.ndarray:
    """
    Compute inverse-frequency sample weights for a binary target.

    Args:
        y: binary labels.

    Returns:
        Sample weights array.
    """
    pos_rate = float(y.mean())
    pos_rate = max(min(pos_rate, 1.0 - 1e-6), 1e-6)

    w_pos = 0.5 / pos_rate
    w_neg = 0.5 / (1.0 - pos_rate)
    return np.where(y.values == 1, w_pos, w_neg)

### Online confirmation (contract)

In [9]:
def simulate_confirmation(y_prob: np.ndarray, threshold: float, right_delay: int) -> np.ndarray:
    """
    Simulate online confirmation: a prediction at time t is confirmed at t+R.

    Args:
        y_prob: predicted probabilities aligned to time t.
        threshold: decision threshold applied at time t.
        right_delay: confirmation delay R.

    Returns:
        confirmed_pred: array aligned to original index where confirmed_pred[t]
            indicates "a swing confirmed at time t" (originated at t-R).
    """
    raw_pred = (y_prob >= threshold).astype(int)
    confirmed = np.zeros_like(raw_pred)

    if right_delay <= 0:
        return raw_pred

    confirmed[right_delay:] = raw_pred[:-right_delay]
    return confirmed

### Metrics helpers (PR-AUC + reports)

In [10]:
def pr_auc(y_true: pd.Series, y_prob: np.ndarray) -> float:
    """
    Compute PR-AUC (Average Precision).

    Args:
        y_true: true labels.
        y_prob: predicted probabilities.

    Returns:
        PR-AUC score.
    """
    return float(average_precision_score(y_true.values, y_prob))


def print_report(y_true: pd.Series, y_prob: np.ndarray, threshold: float, title: str) -> None:
    """
    Print a classification report for a given threshold.

    Args:
        y_true: true labels.
        y_prob: predicted probabilities.
        threshold: threshold for binary decision.
        title: report title.
    """
    y_pred = (y_prob >= threshold).astype(int)

    print("=" * 100)
    print(title)
    print("PR-AUC:", pr_auc(y_true, y_prob))
    print("Threshold:", threshold)
    print("Confusion matrix:\n", confusion_matrix(y_true.values, y_pred))
    print(classification_report(y_true.values, y_pred, digits=4))


def print_structure_rate_report(
    y_true: pd.Series,
    y_prob: np.ndarray,
    threshold: float,
    right_delay: int,
    title: str,
) -> None:
    """
    Print base rate and confirmed prediction rate to judge structure suitability.

    Args:
        y_true: true labels aligned to time t.
        y_prob: predicted probabilities aligned to time t.
        threshold: decision threshold.
        right_delay: confirmation delay.
        title: report title.
    """
    confirmed_pred = simulate_confirmation(y_prob, threshold, right_delay)

    true_rate = float(y_true.mean())
    confirmed_rate = float(confirmed_pred.mean())
    ratio = confirmed_rate / max(true_rate, 1e-12)

    print("-" * 100)
    print(title)
    print("True label rate:", true_rate)
    print("Confirmed pred rate:", confirmed_rate)
    print("Rate ratio (confirmed/true):", ratio)

### Threshold selection

matching confirmed rate

In [11]:
def pick_threshold_by_confirmed_rate(
    y_prob: np.ndarray,
    right_delay: int,
    target_rate: float,
    grid_size: int = 600,
) -> float:
    """
    Pick threshold so that confirmed positive rate matches target_rate.

    Args:
        y_prob: predicted probabilities on validation.
        right_delay: confirmation delay R.
        target_rate: desired confirmed rate (e.g., base label rate).
        grid_size: number of thresholds to test.

    Returns:
        Selected threshold.
    """
    # Search thresholds in upper tail; swing events should be high confidence.
    q_grid = np.linspace(0.70, 0.999, grid_size)
    thresholds = np.unique(np.quantile(y_prob, q_grid))

    best_thr = float(thresholds[0])
    best_err = float("inf")

    for thr in thresholds:
        confirmed = simulate_confirmation(y_prob, float(thr), right_delay)
        rate = float(confirmed.mean())
        err = abs(rate - target_rate)

        if err < best_err:
            best_err = err
            best_thr = float(thr)

    return best_thr

precision floor

In [12]:
def pick_threshold_by_precision_floor(
    y_true: pd.Series,
    y_prob: np.ndarray,
    min_precision: float,
) -> float:
    """
    Pick a threshold that achieves at least min_precision and maximizes recall.

    Args:
        y_true: true labels.
        y_prob: predicted probabilities.
        min_precision: required minimum precision.

    Returns:
        Selected threshold.
    """
    precision, recall, thresholds = precision_recall_curve(y_true.values, y_prob)

    precision = precision[1:]
    recall = recall[1:]

    valid_idx = np.where(precision >= min_precision)[0]
    if len(valid_idx) == 0:
        # If floor is unattainable, pick the threshold with best precision.
        best_idx = int(np.argmax(precision))
        return float(thresholds[best_idx])

    best_idx = valid_idx[int(np.argmax(recall[valid_idx]))]
    return float(thresholds[best_idx])

### Training: HGB + optional calibration

In [13]:
def fit_hgb_with_calibration(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    method: str = "isotonic",
) -> CalibratedClassifierCV:
    """
    Fit HistGradientBoostingClassifier and calibrate probabilities on validation.

    Args:
        X_train: training features.
        y_train: training labels.
        X_valid: validation features.
        y_valid: validation labels.
        method: calibration method ("sigmoid" or "isotonic").

    Returns:
        CalibratedClassifierCV model.
    """
    base_model = build_hgb_model()
    base_model.fit(X_train, y_train, sample_weight=compute_sample_weights(y_train))

    calibrated = CalibratedClassifierCV(base_model, method=method, cv="prefit")
    calibrated.fit(X_valid, y_valid)

    return calibrated

### Train/eval HIGH + LOW the right way

In [14]:
# Validation probabilities are used to set decision thresholds; the test set is still untouched at this stage.

# Train calibrated HGB models
high_model = fit_hgb_with_calibration(X_train, y_high_train, X_valid, y_high_valid, method="isotonic")
low_model = fit_hgb_with_calibration(X_train, y_low_train, X_valid, y_low_valid, method="isotonic")

# Predict on validation
high_valid_prob = high_model.predict_proba(X_valid)[:, 1]
low_valid_prob = low_model.predict_proba(X_valid)[:, 1]

# Target rate = base label rate (можно сделать *0.8 если хочешь еще реже swings)
target_rate_high = float(y_high_train.mean())
target_rate_low = float(y_low_train.mean())

high_threshold = pick_threshold_by_confirmed_rate(
    y_prob=high_valid_prob,
    right_delay=FRACTAL_CFG.right,
    target_rate=target_rate_high,
)
low_threshold = pick_threshold_by_confirmed_rate(
    y_prob=low_valid_prob,
    right_delay=FRACTAL_CFG.right,
    target_rate=target_rate_low,
)

print("HIGH threshold:", high_threshold)
print("LOW threshold:", low_threshold)

# Evaluate on validation (classification view + structure view)
print_report(y_high_valid, high_valid_prob, high_threshold, "HIGH / Calibrated HGB / VALID")
print_structure_rate_report(
    y_true=y_high_valid,
    y_prob=high_valid_prob,
    threshold=high_threshold,
    right_delay=FRACTAL_CFG.right,
    title="HIGH / STRUCTURE RATE / VALID",
)

print_report(y_low_valid, low_valid_prob, low_threshold, "LOW / Calibrated HGB / VALID")
print_structure_rate_report(
    y_true=y_low_valid,
    y_prob=low_valid_prob,
    threshold=low_threshold,
    right_delay=FRACTAL_CFG.right,
    title="LOW / STRUCTURE RATE / VALID",
)


c:\Users\artkh\anaconda3\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
c:\Users\artkh\anaconda3\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


HIGH threshold: 0.2222222222222222
LOW threshold: 0.256544502617801
HIGH / Calibrated HGB / VALID
PR-AUC: 0.30256769541421796
Threshold: 0.2222222222222222
Confusion matrix:
 [[19692   481]
 [  447   263]]
              precision    recall  f1-score   support

           0     0.9778    0.9762    0.9770     20173
           1     0.3535    0.3704    0.3618       710

    accuracy                         0.9556     20883
   macro avg     0.6656    0.6733    0.6694     20883
weighted avg     0.9566    0.9556    0.9561     20883

----------------------------------------------------------------------------------------------------
HIGH / STRUCTURE RATE / VALID
True label rate: 0.033998946511516547
Confirmed pred rate: 0.03562706507685677
Rate ratio (confirmed/true): 1.047887323943662
LOW / Calibrated HGB / VALID
PR-AUC: 0.3129712209684747
Threshold: 0.256544502617801
Confusion matrix:
 [[19698   488]
 [  416   281]]
              precision    recall  f1-score   support

           0     0.9

### Final test evaluation (OOS)

In [15]:
# Run the final out-of-sample check with the frozen thresholds chosen on validation.

high_test_prob = high_model.predict_proba(X_test)[:, 1]
low_test_prob = low_model.predict_proba(X_test)[:, 1]

print_report(y_high_test, high_test_prob, high_threshold, "HIGH / Calibrated HGB / TEST")
print_structure_rate_report(
    y_true=y_high_test,
    y_prob=high_test_prob,
    threshold=high_threshold,
    right_delay=FRACTAL_CFG.right,
    title="HIGH / STRUCTURE RATE / TEST",
)

print_report(y_low_test, low_test_prob, low_threshold, "LOW / Calibrated HGB / TEST")
print_structure_rate_report(
    y_true=y_low_test,
    y_prob=low_test_prob,
    threshold=low_threshold,
    right_delay=FRACTAL_CFG.right,
    title="LOW / STRUCTURE RATE / TEST",
)


HIGH / Calibrated HGB / TEST
PR-AUC: 0.27637714932487745
Threshold: 0.2222222222222222
Confusion matrix:
 [[19631   576]
 [  407   269]]
              precision    recall  f1-score   support

           0     0.9797    0.9715    0.9756     20207
           1     0.3183    0.3979    0.3537       676

    accuracy                         0.9529     20883
   macro avg     0.6490    0.6847    0.6646     20883
weighted avg     0.9583    0.9529    0.9554     20883

----------------------------------------------------------------------------------------------------
HIGH / STRUCTURE RATE / TEST
True label rate: 0.032370827946176316
Confirmed pred rate: 0.040463534932720396
Rate ratio (confirmed/true): 1.25
LOW / Calibrated HGB / TEST
PR-AUC: 0.2870035891156716
Threshold: 0.256544502617801
Confusion matrix:
 [[19662   528]
 [  412   281]]
              precision    recall  f1-score   support

           0     0.9795    0.9738    0.9767     20190
           1     0.3473    0.4055    0.3742      

### Artifact saving

In [ ]:
# Persist everything the runtime detector needs: model files, feature order, thresholds, and metadata.

def save_json(path: Path, payload: Dict) -> None:
    """
    Save a dictionary as JSON.

    Args:
        path: output path.
        payload: dict to save.
    """
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def assert_has_predict_proba(model: object) -> None:
    """
    Assert model has predict_proba method.

    Args:
        model: sklearn-like model.

    Raises:
        ValueError: if predict_proba is missing.
    """
    if not hasattr(model, "predict_proba"):
        raise ValueError("Model does not have predict_proba(). Not a classifier?")


def dump_model(model: object, path: Path) -> None:
    """
    Dump model to disk and validate file size.

    Args:
        model: fitted model.
        path: output path.

    Raises:
        ValueError: if saved file is suspiciously small.
    """
    assert_has_predict_proba(model)
    joblib.dump(model, path)

    size_bytes = os.path.getsize(path)
    print(f"Saved: {path.resolve()} ({size_bytes} bytes)")

    # Very conservative lower bound. Calibrated + base model should be > few KB.
    if size_bytes < 10_000:
        raise ValueError(
            f"Saved model file is too small ({size_bytes} bytes). "
            "Likely not fitted or wrong object saved."
        )


dump_model(high_model, ARTIFACT_PATHS.model_high_path)
dump_model(low_model, ARTIFACT_PATHS.model_low_path)

save_json(
    ARTIFACT_PATHS.feature_config_path,
    {"features": feature_cols, "n_features": len(feature_cols)},
)

save_json(
    ARTIFACT_PATHS.thresholds_path,
    {
        "threshold_high": float(high_threshold),
        "threshold_low": float(low_threshold),
        "left": FRACTAL_CFG.left,
        "right": FRACTAL_CFG.right,
        "selection_rule": "confirmed_rate_matching_on_valid",
        "target_rate_high": float(target_rate_high),
        "target_rate_low": float(target_rate_low),
        "calibration": "isotonic",
    },
)

save_json(
    ARTIFACT_PATHS.meta_path,
    {
        "symbol": "ETHUSDT",
        "timeframe": "15m",
        "labels": f"fractal_L{FRACTAL_CFG.left}_R{FRACTAL_CFG.right}",
        "random_seed": RANDOM_SEED,
        "features_source": str(DATA_PATHS.features_path),
        "labels_source": str(DATA_PATHS.labels_path),
    },
)

print("Saved model files:", [ARTIFACT_PATHS.model_high_path.name, ARTIFACT_PATHS.model_low_path.name])
print(
    "Saved config files:",
    [
        ARTIFACT_PATHS.feature_config_path.name,
        ARTIFACT_PATHS.thresholds_path.name,
        ARTIFACT_PATHS.meta_path.name,
    ],
)
